# Feature 5 — Recall optimization
These notebooks use **real saved logits**, so images, checkpoints, retraining and GPU are unnecessary. Run cells from top to bottom. In Colab, use Runtime → Run all. Your input is `/content/drive/MyDrive/Features_5_6/real_predictions.csv`.

A logit is a model score before conversion to probabilities. Temperature scaling adjusts confidence on each fold's validation set. Thresholds decide when to flag a priority class. Test data is used only to measure the frozen choices.


### 1. Locate your saved scores

In [1]:
# In Colab, use your own writable Drive folder. No GPU is needed.
import os
from pathlib import Path
try:
    from google.colab import drive
except ImportError:
    DATA_DIR = Path(os.environ.get('MELANOMA_DATA_DIR', '.')).resolve()
else:
    drive.mount('/content/drive')
    DATA_DIR = Path('/content/drive/MyDrive/MelanomaClassfierProject/Features_5_6')
INPUT = DATA_DIR / 'real_predictions.csv'
assert INPUT.is_file(), f'Put real_predictions.csv in {DATA_DIR}'


Mounted at /content/drive


### 2. Helpers
Functions check the data, fit temperature, select thresholds, and calculate recall intervals. Calibration lives here so both notebooks use the same frozen probabilities.

In [2]:
import argparse
import hashlib
import json
from pathlib import Path

import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from scipy.optimize import minimize_scalar
from scipy.special import logsumexp, softmax
from sklearn.metrics import (auc, confusion_matrix, precision_recall_curve,
                             precision_recall_fscore_support, roc_curve)

# This order is the CSV contract, not an assumption about the training model.
CLASSES = ['nv', 'mel', 'bkl', 'bcc', 'akiec', 'vasc', 'df']
PRIORITY = ['mel', 'bcc', 'akiec']
ORDER = PRIORITY + [c for c in CLASSES if c not in PRIORITY]
LOGITS = ['logit_' + c for c in CLASSES]
HIGH = [CLASSES.index(c) for c in PRIORITY]
TARGETS = {'mel': .85, 'bcc': .85, 'akiec': .70}


def read_predictions(path):
    """Reject incomplete scores, unknown labels, and duplicate image records."""
    df = pd.read_csv(path)
    required = ['image_id', 'lesion_id', 'dx', 'fold', 'split', 'source', 'model_id'] + LOGITS
    if not set(required) <= set(df.columns):
        raise ValueError('Missing columns: ' + str(sorted(set(required) - set(df.columns))))
    if df[required].isna().any().any() or df.empty:
        raise ValueError('Required fields cannot be empty.')
    if not set(df.dx) <= set(CLASSES) or not set(df.split) <= {'val', 'test'}:
        raise ValueError('Unknown class or split (use val/test).')
    if not np.isfinite(df[LOGITS].to_numpy(float)).all():
        raise ValueError('Logits must be finite numbers.')
    if df.duplicated(['fold', 'split', 'image_id']).any():
        raise ValueError('Duplicate image within a fold/split.')
    if df.groupby('lesion_id').dx.nunique().max() > 1:
        raise ValueError('One lesion has conflicting labels.')
    if set(df.source) not in ({'SIMULATED'}, {'REAL'}):
        raise ValueError('Use one source throughout: SIMULATED or REAL.')
    if df['fold'].nunique() != 5:
        raise ValueError('The primary report requires all five outer folds.')
    for fold, part in df.groupby('fold'):
        if set(part.split) != {'val', 'test'} or part.model_id.nunique() != 1:
            raise ValueError(f'Fold {fold}: need val/test scores from one checkpoint.')
        val = part[part.split == 'val']; test = part[part.split == 'test']
        if set(val.lesion_id) & set(test.lesion_id):
            raise ValueError(f'Fold {fold}: validation/test lesion leakage.')
        if set(val.dx) != set(CLASSES) or set(test.dx) != set(CLASSES):
            raise ValueError(f'Fold {fold}: all seven classes required in val/test.')
    test = df[df.split == 'test']
    if test.image_id.duplicated().any() or test.groupby('lesion_id')['fold'].nunique().max() > 1:
        raise ValueError('Each image/lesion must belong to only one outer test fold.')
    return df


def labels(df):
    return df.dx.map({c: i for i, c in enumerate(CLASSES)}).to_numpy(int)


def fit_temperature(logits, y):
    """Learn one positive temperature on validation by minimizing log loss.

    Temperature rescales confidence; it does not change the top-ranked class.
    Optimize log(T) to prevent an invalid zero/negative temperature.
    """
    def loss(log_t):
        scaled = logits / np.exp(log_t)
        return np.mean(logsumexp(scaled, axis=1) - scaled[np.arange(len(y)), y])
    result = minimize_scalar(loss, bounds=(-4, 4), method='bounded')
    if not result.success:
        raise RuntimeError('Temperature fitting failed.')
    return float(np.exp(result.x))


def choose_threshold(y_binary, scores, min_specificity):
    """Maximize validation recall subject to the requested specificity.

    Ties prefer higher specificity, then the higher (more conservative) cutoff.
    A cutoff just above the largest score permits predicting no positives.
    """
    fpr, tpr, thresholds = roc_curve(y_binary, scores, drop_intermediate=False)
    thresholds[~np.isfinite(thresholds)] = np.nextafter(scores.max(), np.inf)
    eligible = np.flatnonzero(1 - fpr >= min_specificity - 1e-12)
    best = max(eligible, key=lambda i: (tpr[i], -fpr[i], thresholds[i]))
    return float(thresholds[best])


def binary_metrics(actual, predicted):
    tn, fp, fn, tp = confusion_matrix(actual, predicted, labels=[False, True]).ravel()
    return {'recall': float(tp / (tp + fn)) if tp + fn else None,
            'specificity': float(tn / (tn + fp)) if tn + fp else None,
            'precision': float(tp / (tp + fp)) if tp + fp else 0.,
            'tp': int(tp), 'fp': int(fp), 'tn': int(tn), 'fn': int(fn)}


def ece(y, probs, bins=15):
    """Top-label ECE: weighted confidence/accuracy gaps in equal-width bins."""
    confidence = probs.max(axis=1)
    correct = probs.argmax(axis=1) == y
    bucket = np.minimum((confidence * bins).astype(int), bins - 1)
    gap = 0.; rows = []
    for b in range(bins):
        mask = bucket == b
        if mask.any():
            acc = float(correct[mask].mean()); conf = float(confidence[mask].mean())
            gap += mask.mean() * abs(acc - conf)
            rows.append((conf, acc, int(mask.sum())))
    return float(gap), rows


def grouped_recall_ci(y, pred, lesion_ids, iterations=1000, seed=42):
    """Percentile bootstrap over whole lesions, retaining all their images.

    Per-lesion counts make this efficient. Image-level recall is estimated;
    related images are never independently resampled. Absent classes are skipped.
    """
    groups, inverse = np.unique(lesion_ids, return_inverse=True)
    totals = np.zeros((len(groups), 7)); hits = totals.copy()
    np.add.at(totals, (inverse, y), 1)
    np.add.at(hits, (inverse, y), pred == y)
    rng = np.random.default_rng(seed)
    samples = np.full((iterations, 7), np.nan)
    for b in range(iterations):
        draw = rng.integers(0, len(groups), len(groups))
        den = totals[draw].sum(axis=0)
        samples[b] = np.divide(hits[draw].sum(axis=0), den,
                               out=np.full(7, np.nan), where=den > 0)
    return np.nanpercentile(samples, [2.5, 97.5], axis=0)


def binary_recall_ci(truth, predicted, lesion_ids, iterations=2000, seed=42):
    # Resample entire lesions, preserving all images of each sampled lesion.
    groups, inverse = np.unique(lesion_ids, return_inverse=True)
    positives = np.bincount(inverse, weights=np.asarray(truth, int), minlength=len(groups))
    hits = np.bincount(inverse, weights=np.asarray(truth) & np.asarray(predicted), minlength=len(groups))
    rng = np.random.default_rng(seed)
    values = []
    for _ in range(iterations):
        draw = rng.integers(len(groups), size=len(groups))
        denominator = positives[draw].sum()
        if denominator:
            values.append(hits[draw].sum() / denominator)
    return np.percentile(values, [2.5, 97.5]).tolist()


### 3. Fit on validation, report on held-out tests
Maximize recall subject to validation specificity ≥80%. Ties prefer higher specificity, then higher cutoff. No test results affect these choices.

In [3]:
df = read_predictions(INPUT)
assert set(df.source) == {'REAL'}, 'This notebook requires real model scores.'
OUT = DATA_DIR / 'Feature_5'
OUT.mkdir(parents=True, exist_ok=True)
settings = {}; held_out = []; validation_rows = []
for fold, data in df.groupby('fold', sort=True):
    val = data[data.split == 'val']; test = data[data.split == 'test'].copy()
    vy = labels(val)
    # All fitted parameters below use validation data ONLY.
    temperature = fit_temperature(val[LOGITS].to_numpy(float), vy)
    vp = softmax(val[LOGITS].to_numpy(float) / temperature, axis=1)
    thresholds = {}
    for c in PRIORITY:
        i = CLASSES.index(c)
        thresholds[c] = choose_threshold(vy == i, vp[:, i], .80)
        validation_rows.append({'fold': int(fold), 'class': c, 'threshold': thresholds[c],
                               **binary_metrics(vy == i, vp[:, i] >= thresholds[c])})
    risk_threshold = choose_threshold(np.isin(vy, HIGH), vp[:, HIGH].sum(axis=1), .90)
    settings[str(fold)] = {'model_id': str(data.model_id.iloc[0]), 'temperature': temperature,
        'priority_thresholds': thresholds, 'high_risk_threshold': risk_threshold}
    tp = softmax(test[LOGITS].to_numpy(float) / temperature, axis=1)
    raw = softmax(test[LOGITS].to_numpy(float), axis=1)
    for i, c in enumerate(CLASSES):
        test['prob_' + c] = tp[:, i]
        test['raw_prob_' + c] = raw[:, i]
    test['top_class'] = np.array(CLASSES)[tp.argmax(axis=1)]
    for c in PRIORITY:
        test['flag_' + c] = test['prob_' + c] >= thresholds[c]
    test['high_risk_flag'] = tp[:, HIGH].sum(axis=1) >= risk_threshold
    held_out.append(test)
test = pd.concat(held_out, ignore_index=True)
test.to_csv(OUT / 'out_of_fold_predictions.csv', index=False)
pd.DataFrame(validation_rows).to_csv(OUT / 'validation_thresholds.csv', index=False)
manifest = {'input_sha256': hashlib.sha256(INPUT.read_bytes()).hexdigest(),
            'class_order': CLASSES, 'folds': settings,
            'oof_sha256': hashlib.sha256((OUT / 'out_of_fold_predictions.csv').read_bytes()).hexdigest()}
(OUT / 'fold_settings.json').write_text(json.dumps(manifest, indent=2))
rows = []; fold_rows = []
for c in PRIORITY:
    truth = test.dx.to_numpy() == c
    flagged = test['flag_' + c].to_numpy(bool)
    m = binary_metrics(truth, flagged)
    lo, hi = binary_recall_ci(truth, flagged, test.lesion_id.to_numpy())
    rows.append({'class': c, **m, 'recall_ci_low': lo, 'recall_ci_high': hi,
                 'recall_target': TARGETS[c],
                 'meets_test_targets': m['recall'] >= TARGETS[c] and m['specificity'] >= .80})
    for fold, part in test.groupby('fold'):
        fold_rows.append({'fold': int(fold), 'class': c,
                          **binary_metrics(part.dx == c, part['flag_' + c])})
priority = pd.DataFrame(rows)
priority.to_csv(OUT / 'priority_threshold_metrics.csv', index=False)
pd.DataFrame(fold_rows).to_csv(OUT / 'priority_metrics_by_fold.csv', index=False)
print('REAL held-out results: validation-selected class flags')
print(priority.round(4).to_string(index=False))
print('Saved settings and predictions to:', OUT)


REAL held-out results: validation-selected class flags
class  recall  specificity  precision  tp   fp   tn  fn  recall_ci_low  recall_ci_high  recall_target  meets_test_targets
  mel  0.7691       0.8024     0.3273 856 1759 7143 257         0.7391          0.8000           0.85               False
  bcc  0.9514       0.8401     0.2435 489 1519 7982  25         0.9290          0.9720           0.85                True
akiec  0.9572       0.8416     0.1694 313 1535 8153  14         0.9288          0.9803           0.70                True
Saved settings and predictions to: /content/drive/MyDrive/MelanomaClassfierProject/Features_5_6/Feature_5
